# Task B -- full-data R-Drop submission with multiclass-augmented binary data

This notebook trains the R-Drop version of the winning Task B recipe (Run 9, which scored
**0.6410 macro-F1** and **0.6937 accuracy** on CodaBench) augmented with the additional
multiclass-labeled hate comments compiled from the binary corpus (`hate_only_compiled_multiclass.csv`).

Recipe:
- TAPT MuRIL on all permitted Task B text, external OffensEval Kannada text, and augmented data
- one encoder layer reinitialized (`--reinit-layers 1`)
- R-Drop KL weight 0.5 (`--rdrop 0.5`)
- five classifier seeds: 42, 43, 44, 45, 46
- all labelled rows (official gold `multiclass_train.csv` + new non-overlapping rows from `hate_only_compiled_multiclass.csv`)
- six epochs, mean+max pooling, FGM, EMA, balanced class weighting, no auxiliary head

The five validation probability matrices are averaged, then converted to the required
395-row id,label file. The final validated file is
`b_reinit1_rdrop_augmented_full.zip`. Upload that ZIP to the Task B CodaBench phase after
reviewing the log. Expected runtime is approximately 3--5 hours on a T4 x2 or P100.


In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
pathlib.Path("artifacts/data").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Prepare and inspect augmented training dataset

`multiclass_train.csv` contains 3,159 official gold-labeled rows.
`data/external/hate_only_compiled_multiclass.csv` contains 3,545 multiclass-labeled comments derived from the binary hate subset.
We merge them giving precedence to the official gold labels:
- All 3,159 official gold rows are preserved unchanged.
- The 779 new rows not in `multiclass_train.csv` are appended.
The augmented dataset (3,938 rows) is saved to `artifacts/data/multiclass_train_augmented.csv`.


In [ ]:
import pandas as pd

mc_train = pd.read_csv("data/raw/multiclass_train.csv")
ext_path = pathlib.Path("data/external/hate_only_compiled_multiclass.csv")
if not ext_path.exists():
    for candidate in [pathlib.Path("/kaggle/input/hate_only_compiled_multiclass.csv"),
                      pathlib.Path.home() / "Downloads/hate_only_compiled_multiclass.csv"]:
        if candidate.exists():
            ext_path = candidate
            break

ext_df = pd.read_csv(ext_path)
if "Hate Category" not in ext_df.columns and "Label" in ext_df.columns:
    ext_df = ext_df.rename(columns={"Label": "Hate Category"})

print(f"multiclass_train rows: {len(mc_train)}")
print(f"external hate_only_compiled_multiclass rows: {len(ext_df)}")

# Gold-priority merge: keep all official rows, add unseen IDs
new_rows = ext_df[~ext_df["id"].isin(mc_train["id"])]
augmented_train = pd.concat([mc_train, new_rows], ignore_index=True)

AUG_DATA_PATH = "artifacts/data/multiclass_train_augmented.csv"
augmented_train.to_csv(AUG_DATA_PATH, index=False)

print(f"\naugmented_train rows: {len(augmented_train)} ({len(new_rows)} new rows appended)")
print("\nClass distribution in augmented training set:")
print(augmented_train["Hate Category"].value_counts())


## 2. Run full-data TAPT

The TAPT input is the complete permitted corpus: the augmented training dataset plus
`offenseval_kn.csv`. Labels are not used by the masked-language-model stage. The min-words
and deduplication flags match the Run 9 submission.


In [ ]:
TAPT_OUT = "artifacts/runs/tapt-d0v0-rdrop-augmented-full"
TAPT_LOG = "artifacts/logs/rdrop_augmented_full_tapt.log"
if (pathlib.Path(TAPT_OUT) / "config.json").exists():
    print("using existing TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--corpus", AUG_DATA_PATH,
         "data/external/offenseval_kn.csv",
         "--val-frac", "0", "--min-words", "1", "--no-dedupe",
         "--out", TAPT_OUT], log=TAPT_LOG)
assert (pathlib.Path(TAPT_OUT) / "config.json").exists(), "TAPT checkpoint was not written"
print("full-data TAPT checkpoint ready:", TAPT_OUT)


## 3. Train five full-data R-Drop models

folds=1 means there is no validation split: each seed trains on every augmented labelled row and
contributes validation-input probabilities to the five-seed average. This matches the Run 9
0.6410 recipe with 1-layer reinitialization, R-Drop 0.5, 6 epochs, and seeds 42--46.


In [ ]:
TAG = "b_reinit1_rdrop_augmented_full"
run_dir = pathlib.Path("artifacts/runs") / TAG
run([sys.executable, "-u", "-m", "hastika.task_b.train",
     "--tag", TAG,
     "--train-data", AUG_DATA_PATH,
     "--model", TAPT_OUT,
     "--folds", "1",
     "--no-dedupe",
     "--reinit-layers", "1",
     "--rdrop", "0.5",
     "--aux-weight", "0",
     "--seeds", "42", "43", "44", "45", "46",
     "--epochs", "6"],
    log=f"artifacts/logs/{TAG}.log")
assert (run_dir / "predictions.csv").exists(), "full-data predictions were not written"
assert (run_dir / "test_probs.npy").exists(), "full-data probabilities were not written"
print("five-seed full-data R-Drop fit completed:", run_dir)


## 4. Validate and package the CodaBench submission

The submission helper checks the 395 validation IDs, allowed six-way labels, and exact
id,label header before writing a ZIP containing one bare predictions.csv.


In [ ]:
PRED = pathlib.Path("artifacts/runs") / TAG / "predictions.csv"
ZIP = pathlib.Path("/kaggle/working") / f"{TAG}.zip"
run([sys.executable, "-m", "hastika.common.submission",
     "--task", "b", "--pred", str(PRED), "--out", str(ZIP)])

with zipfile.ZipFile(ZIP) as z:
    assert z.namelist() == ["predictions.csv"], z.namelist()
print("READY TO UPLOAD:", ZIP)


## 5. Preserve the submission output

Download the output directory from Kaggle. The ZIP is the only file needed for CodaBench;
the log, prediction CSV, probability matrix, and TAPT log are included for reproducibility.


In [ ]:
OUT = pathlib.Path("/kaggle/working/rdrop_augmented_submission")
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / ZIP.name)
shutil.copy2(PRED, OUT / PRED.name)
shutil.copy2(pathlib.Path("artifacts/runs") / TAG / "test_probs.npy",
             OUT / f"{TAG}_test_probs.npy")
shutil.copy2(pathlib.Path("artifacts/logs") / f"{TAG}.log",
             OUT / f"{TAG}.log")
if pathlib.Path(TAPT_LOG).exists():
    shutil.copy2(TAPT_LOG, OUT / pathlib.Path(TAPT_LOG).name)
print("download:", OUT / ZIP.name)
print("all reproducibility files:", OUT)
